# MeatLens MobileNetV3Small Segmented 6-Fold Hybrid Notebook (Latest Loader)

Use this fresh notebook if the older notebook keeps executing stale code in the kernel or notebook cache.


This notebook prepares the new MeatLens segmented ROI experiment:

- MobileNetV3Small only
- 6-fold cross-rotation only
- already-preprocessed HSV/LAB segmented `224x224` ROI images
- hybrid CNN + color/texture features
- new output root only: `training_outputs/mobilenetv3small_segmented6_hybrid/`

Important preprocessing rule:
- training does **not** apply `center_square_resize_224`
- the segmented ROI images are loaded directly
- resizing happens only as a safety fallback when an image is not already `224x224`


Feature note:

The features are extracted from the segmented ROI image, including the neutral/black/gray background introduced by segmentation.


Transition-aware evaluation note:

Strict accuracy and macro F1 remain the primary evaluation metrics. Transition-aware metrics are added only as secondary analysis because pork freshness changes gradually and borderline images may visually fall between adjacent classes.


In [ ]:
# ============================================================
# Load the segmented6 hybrid library from the .py source file
# This avoids stale embedded notebook code and ensures the
# latest library version is what the kernel executes.
# ============================================================


In [ ]:
import importlib
import inspect

import pandas as pd
from IPython.display import Image as DisplayImage, display

import mobilenetv3small_segmented6_hybrid_lib as segmented6_lib

segmented6_lib = importlib.reload(segmented6_lib)

for _name in dir(segmented6_lib):
    if not _name.startswith("_"):
        globals()[_name] = getattr(segmented6_lib, _name)

print("Library source:", segmented6_lib.__file__)
print(
    "Function first lines:",
    {
        "train_segmented6_hybrid_model": train_segmented6_hybrid_model.__code__.co_firstlineno,
        "run_single_segmented6_hybrid_experiment": run_single_segmented6_hybrid_experiment.__code__.co_firstlineno,
    },
)
print(
    "Function source files:",
    {
        "train_segmented6_hybrid_model": inspect.getsourcefile(train_segmented6_hybrid_model),
        "run_single_segmented6_hybrid_experiment": inspect.getsourcefile(run_single_segmented6_hybrid_experiment),
    },
)

sanity_optimizer = make_optimizer(1e-3)
print("Sanity optimizer type:", type(sanity_optimizer))
print("Sanity optimizer module:", type(sanity_optimizer).__module__)

if inspect.getsourcefile(run_single_segmented6_hybrid_experiment) != segmented6_lib.__file__:
    raise RuntimeError("Notebook is not using the module-backed experiment function.")

if "legacy" not in type(sanity_optimizer).__module__:
    raise RuntimeError("Notebook did not load the DirectML-safe legacy Adam optimizer.")

print("Notebook sanity check passed.")

ensure_output_dirs()
print_library_status()


In [ ]:
split_dfs = load_all_cross_rotation_splits()
metadata_validation_df = validate_metadata_mapping(split_dfs)
metadata_validation_df


In [ ]:
quality_bundle = build_dataset_quality_summary(split_dfs)
print_quality_tables(quality_bundle)
quality_bundle["summary_df"].head(20)


In [ ]:
sample_image_figure_path = save_sample_visualization(split_dfs)
print("Saved sample image figure:", sample_image_figure_path)
display(DisplayImage(filename=str(sample_image_figure_path)))


In [ ]:
smoke_features_df = run_feature_extraction_smoke_test(split_dfs)
smoke_features_df


In [ ]:
print("Dataset quality summary CSV:", SEGMENTED6_QUALITY_SUMMARY_PATH)
print("Feature smoke test CSV:", EXTENSION_FEATURES_ROOT / "example_3_segmented_roi_color_texture_features.csv")
print("Segmented 6-fold split root:", SEGMENTED_SPLITS_ROOT)
print("Output root:", EXTENSION_OUTPUT_ROOT)


In [ ]:
if RUN_EXTENSION_TRAINING:
    all_results = run_segmented6_hybrid_training()
else:
    print("Segmented 6-fold hybrid training functions are ready. Set RUN_EXTENSION_TRAINING=True to train.")


In [ ]:
regenerate_bundle = regenerate_transition_metrics_and_graphs()


In [ ]:
if SEGMENTED6_SEED_METRICS_PATH.exists():
    seed_metrics_df = pd.read_csv(SEGMENTED6_SEED_METRICS_PATH)
    prediction_df = load_prediction_csvs_for_metrics(seed_metrics_df)
    print(build_summary_interpretation_text(seed_metrics_df, prediction_df))
else:
    print("No segmented6 hybrid training results found yet.")


In [ ]:
debug_result = run_single_segmented6_hybrid_experiment(
     fold_name="fold1",
     seed=42,
 )
